<b><font size="6" color="#E8800A">Week 12a: Neural networks I</font></b><br>
<b><font size="4">A prediction through the hidden layer</font></b><br>

A hidden layer transforms features before the output combines them, so the activation determines whether a network can bend its decision boundary. We compute that transformation by hand, then measure how activation, depth and width, scale and iteration budget change the fitted network on the course folds.

<div class="alert alert-block alert-info">

# TOC<a class="anchor" id="toc"></a>
* [<font color='#E8800A'>Setup</font>](#setup)
* [<font color='#E8800A'>The network by hand</font>](#toy)
* [<font color='#E8800A'>The MLP reference on the course data</font>](#reference)
* [<font color='#E8800A'>MLP settings</font>](#mlp_settings)
* [<font color='#E8800A'>The course board</font>](#board)
* [<font color='#E8800A'>Key takeaways</font>](#takeaways)
* [<font color='#E8800A'>Exit ticket</font>](#exit)
* [<font color='#E8800A'>References</font>](#references)

</div>

# <font color='#E8800A'>Setup</font> <a class="anchor" id="setup"></a>
[Back to TOC](#toc)

In [ ]:
import sys
import json
import joblib
from sklearn.pipeline import Pipeline
from pathlib import Path
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

from copy import deepcopy
import time
import warnings
from functools import partial
from joblib import parallel_config
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from matplotlib_inline.backend_inline import set_matplotlib_formats
from sklearn.datasets import make_moons
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm
from course_helpers import PLOT_BLUE, PLOT_ORANGE, Scoreboard, record, results_before, result_context

random_state = 33
np.random.seed(random_state)
sns.set_theme(style="whitegrid")
set_matplotlib_formats("svg")
plt.rcParams["figure.dpi"] = 150

ignore_unknown_category_warning = partial(
    warnings.filterwarnings, "ignore",
    message=r"^Found unknown categories in columns \[.*\] during transform\. These unknown categories will be encoded as all zeros$",
    category=UserWarning, module=r"^sklearn\.preprocessing\._encoders$")
ignore_unknown_category_warning()
def unscaled_preparation_warning_policy():
    ignore_unknown_category_warning()
    warnings.filterwarnings("ignore",
        message=r"^lbfgs failed to converge after 1000 iteration\(s\) \(status=1\):\nSTOP: TOTAL NO\. OF ITERATIONS REACHED LIMIT\n\nIncrease the number of iterations to improve the convergence \(max_iter=1000\)\.\nYou might also want to scale the data as shown in:\n    https://scikit-learn\.org/stable/modules/preprocessing\.html\nPlease also refer to the documentation for alternative solver options:\n    https://scikit-learn\.org/stable/modules/linear_model\.html#logistic-regression$",
        category=ConvergenceWarning, module=r"^sklearn\.linear_model\._logistic$")

from preprocessing import (
    FoldPreprocessor,
    load_recipe,
    outer_folds,
    outer_splitter,
    prepared_folds,
)
classification_recipe_log = "../../logs/week_04_feature_work_classification_log.json"


__Step 1:__ Load the frame, set aside the test rows, and prepare the training folds.

In [ ]:
# The champions table with the feature recipe of the feature-work notebook, indexed by athlete.
data, classification_recipe = load_recipe(
    "../../data/interim/champions.csv", classification_recipe_log)
data = data.set_index("RecordID")
X = data[classification_recipe["categorical"] + classification_recipe["numeric"]]
y = data["Outcome"]
# The test rows are sealed once: 20 percent of the athletes, stratified by outcome, and no choice below reads them.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=random_state)
mlp_folds = outer_splitter("classification", random_state=random_state)
# `mlp_prepared` holds, for each of the ten folds, the rows to fit (`X_fit`, `y_fit`) and the rows to score (`X_score`, `y_score`), prepared inside the fold. The recipe is also fitted once on all the training rows, so every model fitted on all the rows learns from `Z_train` and is scored on `Z_test`.
with parallel_config(backend="loky", initializer=ignore_unknown_category_warning):
    mlp_prep = FoldPreprocessor(classification_recipe).fit(X_train, y_train)
    Z_train = mlp_prep.transform(X_train)
    Z_test = mlp_prep.transform(X_test)
    mlp_prepared = prepared_folds(FoldPreprocessor(classification_recipe), mlp_folds,
                                  X_train, y_train, n_jobs=4)
print(f"{len(y_train):,} training athletes, {len(y_test):,} test athletes; {mlp_folds.get_n_splits()} validation folds")

# <font color='#E8800A'>The network by hand</font> <a class="anchor" id="toy"></a>
[Back to TOC](#toc)

The moons generated by `make_moons` are two interlocking curves, so separating them requires a curved boundary. The network below keeps the scikit-learn defaults: one hidden layer of 100 ReLU units, the Adam optimiser, an L2 penalty `alpha=0.0001` and at most 200 iterations. The same settings are the starting point on the real data.

__Step 2:__ Split the moons and standardise their training coordinates.

In [ ]:
moon_X, moon_y = make_moons(n_samples=600, noise=0.2, random_state=random_state)
moon_train, moon_test, moon_y_train, moon_y_test = train_test_split(
    moon_X, moon_y, test_size=0.2, stratify=moon_y, random_state=random_state)
moon_scaler = StandardScaler().fit(moon_train)
moon_train_ready = moon_scaler.transform(moon_train)
moon_test_ready = moon_scaler.transform(moon_test)

__Step 3:__ Fit the default network on the toy training rows. Use `MLPClassifier(**fixed_settings)`.

In [ ]:
# The library defaults for everything but the seed.
fixed_settings = {"random_state": random_state}
toy_network = ....fit(moon_train_ready, moon_y_train)
print(f"toy held-out accuracy: {accuracy_score(moon_y_test, toy_network.predict(moon_test_ready)):.4f}")

__Step 4:__ Read `coefs_`, the weight matrices.

In [ ]:
print([array.shape for array in toy_network.coefs_])

__Step 5:__ Read `intercepts_`, the bias vectors.

In [ ]:
print([array.shape for array in toy_network.intercepts_])

__Step 6:__ Read `n_iter_`, the iterations completed.

In [ ]:
print(toy_network.n_iter_)

__Step 7:__ Read `loss_`, the final training objective.

In [ ]:
print(toy_network.loss_)

**The default toy network ends at loss 0.2171 after 200 iterations and raises a ConvergenceWarning.** Its weight matrices have shapes (2, 100) and (100, 1), with bias vectors of lengths 100 and 1.

__Step 8:__ Read the fitted arrays and the activation function of each layer, and choose the units to draw.

In [ ]:
# The fitted network's own arrays. A layer wider than six units shows its first five units, an ellipsis and its last unit.
net_model = toy_network
net_weights, net_biases = net_model.coefs_, net_model.intercepts_
net_sizes = [net_weights[0].shape[0]] + [weight.shape[1] for weight in net_weights]
net_shown = [list(range(size)) if size <= 6 else [*range(5), None, size - 1] for size in net_sizes]
# The activation of each layer is read from the fitted model: `activation` for the hidden layers and `out_activation_` for the output.
net_activations = ["none"] + [net_model.activation] * (len(net_sizes) - 2) + [net_model.out_activation_]
net_symbol = "p"
net_layers = pd.DataFrame({"units": net_sizes, "units drawn": [len([unit for unit in shown if unit is not None]) for shown in net_shown],
                           "activation": net_activations},
                          index=["input"] + [f"hidden {layer}" for layer in range(1, len(net_sizes) - 1)] + ["output"])
display(net_layers)

__Step 9:__ Place the drawn units and name their nodes.

In [ ]:
# One row per unit drawn, and one row per connection between units drawn in neighbouring layers.
net_gap = 4
net_prefix = ["x"] + (["h"] if len(net_sizes) == 3 else [f"h{layer}_" for layer in range(1, len(net_sizes) - 1)])
net_math = ["x"] + (["h"] if len(net_sizes) == 3 else [f"h^{{({layer})}}" for layer in range(1, len(net_sizes) - 1)])
net_node_rows = []
for layer, slots in enumerate(net_shown):
    last = layer == len(net_sizes) - 1
    for slot, unit in enumerate(slots):
        net_node_rows.append({"layer": layer, "unit": -1 if unit is None else unit, "x": layer * net_gap, "y": 1.2 * ((len(slots) - 1) / 2 - slot),
                              "name": "output" if last else f"{net_prefix[layer]}{unit + 1}" if unit is not None else "",
                              "tex": f"${net_symbol}$" if last else f"${net_math[layer]}_{{{unit + 1}}}$" if unit is not None else ""})
net_nodes = pd.DataFrame(net_node_rows)
net_circles = net_nodes[net_nodes["unit"] >= 0]


__Step 10:__ Extract the fitted weights connecting the drawn units.

In [ ]:
net_link_rows = []
for layer, weights in enumerate(net_weights):
    for source in net_circles[net_circles["layer"] == layer].itertuples():
        for target in net_circles[net_circles["layer"] == layer + 1].itertuples():
            net_link_rows.append({"from": source.name, "to": target.name, "weight": weights[source.unit, target.unit],
                                  "x0": source.x, "y0": source.y, "x1": target.x, "y1": target.y})
net_links = pd.DataFrame(net_link_rows)
net_largest = max(np.abs(weight).max() for weight in net_weights)


__Step 11:__ Label each layer with its fitted activation and array shapes.

In [ ]:
# The text of the figure: a title for each layer with its activation function, and the shapes of each pair of arrays.
net_hidden_formula = {"relu": r"\max(0,z)", "tanh": r"\tanh(z)", "logistic": r"\sigma(z)", "identity": "z"}[net_model.activation]
net_output_formula = {"logistic": r"\sigma(a)", "identity": "a"}[net_model.out_activation_]
net_hidden_names = ["hidden layer" if len(net_sizes) == 3 else f"hidden layer {layer}" for layer in range(1, len(net_sizes) - 1)]
net_titles = ([f"inputs $x$\n{net_sizes[0]} units"]
              + [f"{name}, {size} units\n{net_model.activation}: $h={net_hidden_formula}$" for name, size in zip(net_hidden_names, net_sizes[1:-1])]
              + [f"output\n{net_model.out_activation_}: ${net_symbol}={net_output_formula}$" + ""])
net_captions = [f"$W_{{{k}}}$ ({w.shape[0]}×{w.shape[1]}), $b_{{{k}}}$ ({len(b)})" for k, (w, b) in enumerate(zip(net_weights, net_biases), 1)]
net_face = ["#EDEDED"] + ["#FFF3C4"] * (len(net_sizes) - 2) + ["#DFF5E8"]
print(f"{len(net_links)} connections drawn; largest absolute weight of the network {net_largest:.3f}")
# The ten largest of the connections drawn.
display(net_links.assign(size=net_links["weight"].abs()).nlargest(10, "size")[["from", "to", "weight"]].round(3).reset_index(drop=True))

__Step 12:__ Draw the network from that table.

In [ ]:
net_blue, net_orange = "#1779A8", "#C96A00"
net_fig, net_ax = plt.subplots(figsize=(min(14, 2.6 * len(net_sizes) + 2), 7.4))
for link in net_links.itertuples():
    net_size = np.abs(link.weight) / net_largest
    net_ax.plot([link.x0, link.x1], [link.y0, link.y1], color=net_blue if link.weight > 0 else net_orange,
                lw=0.4 + 4.6 * net_size, alpha=0.12 + 0.8 * net_size, solid_capstyle="round", zorder=1)
net_ax.scatter(net_circles["x"], net_circles["y"], s=560, facecolor=[net_face[layer] for layer in net_circles["layer"]],
               edgecolor="#222222", linewidth=1.2, zorder=3)
for node in net_nodes.itertuples():
    net_ax.text(node.x, node.y, node.tex if node.unit >= 0 else r"$\vdots$", ha="center", va="center",
                fontsize=11 if node.unit >= 0 else 18, zorder=4)
net_top, net_bottom = net_nodes["y"].max(), net_nodes["y"].min()
for layer, title in enumerate(net_titles):
    net_ax.text(layer * net_gap, net_top + 0.9, title, ha="center", va="bottom", fontweight="bold", fontsize=12)
for gap, caption in enumerate(net_captions):
    net_ax.text((gap + 0.5) * net_gap, net_bottom - 1.0, caption, ha="center", va="top", fontsize=12)
# Two keys: the sign, and three sample widths.
net_sign_key = net_ax.legend(handles=[net_ax.plot([], [], color=net_blue, lw=3, label="positive")[0], net_ax.plot([], [], color=net_orange, lw=3, label="negative")[0]],
                             title="weight sign", loc="lower left", bbox_to_anchor=(0.0, -0.06), ncol=2, frameon=False, fontsize=11, title_fontsize=11)
net_ax.add_artist(net_sign_key)
net_ax.legend(handles=[net_ax.plot([], [], color="#555555", lw=0.4 + 4.6 * size, label=f"{size * net_largest:.2f}")[0] for size in (0.25, 0.5, 1.0)],
              title="weight size", loc="lower right", bbox_to_anchor=(1.0, -0.06), ncol=3, frameon=False, fontsize=11, title_fontsize=11)
net_ax.set(xlim=(-1, net_gap * (len(net_sizes) - 1) + 1.4), ylim=(net_bottom - 2.8, net_top + 3.4), title="The fitted moons network")
net_ax.axis("off")
plt.tight_layout()
plt.show()

**The two fitted weight matrices have shapes (2, 100) and (100, 1), and the largest absolute weight is 1.014.** The drawing shows six of the 100 hidden units, with an ellipsis for the omitted units. ReLU changes each hidden sum before the logistic output converts its final sum into a probability.

For one prepared row, the hidden and output calculations are

$$z=xW_1+b_1,\qquad h=\max(0,z),\qquad a=hW_2+b_2,\qquad p=\sigma(a).$$

Here $x$ is the row of inputs, $W_1$ and $W_2$ are the two weight matrices, and $b_1$ and $b_2$ are their bias vectors. The vector $z$ contains hidden weighted sums and $h$ contains their ReLU outputs. The scalar $a$ is the output weighted sum. The function $\sigma(a)=1/(1+e^{-a})$ converts it to class-one probability $p$; $e$ is the exponential base.

__Step 13:__ Compute the hidden weighted sums for one row.

In [ ]:
x = moon_test_ready[[0]]
W1, W2 = toy_network.coefs_
b1, b2 = toy_network.intercepts_
z = x @ W1 + b1
print(z.round(3))

__Step 14:__ Apply ReLU to those sums.

In [ ]:
h = np.maximum(0, z)
print(h.round(3))

__Step 15:__ Convert the output sum into a probability and compare the library result.

In [ ]:
a = h @ W2 + b2
p = 1 / (1 + np.exp(-a))
print(f"by hand: {p.item():.10f}")
print(f"predict_proba: {toy_network.predict_proba(x)[0, 1]:.10f}")
np.testing.assert_allclose(p.item(), toy_network.predict_proba(x)[0, 1])

**The hand calculation gives probability 0.1457395953, the same value as `predict_proba`.** The two calculations use the same fitted weights, biases and prepared input row.

The `activation` parameter changes hidden layers only; binary classification keeps its logistic output. ReLU sets negative weighted sums to zero. Logistic and tanh flatten towards their extremes, while identity returns its input. With identity, $h=xW_1+b_1$, so the output sum becomes $x(W_1W_2)+(b_1W_2+b_2)$. The product $W_1W_2$ is one effective coefficient vector, and $b_1W_2+b_2$ is its effective bias. Additional linear layers still produce an affine score, a weighted sum plus bias. Applying the logistic output gives the same functional form as binary logistic regression. Different optimisation and regularisation can still produce different fitted probabilities.

In [ ]:
activation_input = np.linspace(-4, 4, 401)
activation_values = {"identity": activation_input, "logistic": 1 / (1 + np.exp(-activation_input)),
                     "tanh": np.tanh(activation_input), "relu": np.maximum(0, activation_input)}
activation_formulas = {"identity": r"$h(z)=z$", "logistic": r"$h(z)=1/(1+e^{-z})$",
                       "tanh": r"$h(z)=\tanh(z)$", "relu": r"$h(z)=\max(0,z)$"}
fig, axes = plt.subplots(1, 4, figsize=(12, 3), sharex=True)
for ax, (label, values) in zip(axes, activation_values.items()):
    ax.plot(activation_input, values, linewidth=2)
    ax.axhline(0, color="grey", linewidth=0.7)
    ax.axvline(0, color="grey", linewidth=0.7)
    ax.set(title=label + "\n" + activation_formulas[label], xlabel="Weighted sum z", xlim=(-4, 4))
    ax.set_ylim({"identity": (-4.3, 4.3), "logistic": (-0.1, 1.1),
                 "tanh": (-1.1, 1.1), "relu": (-0.3, 4.3)}[label])
axes[0].set_ylabel("Hidden activation h(z)")
fig.suptitle("Hidden activations: each panel has its own output scale")
plt.tight_layout()
plt.show()

**Identity preserves a straight line; the other curves change its shape.** The logistic output remains present for binary classification even when the hidden activation is identity.

__Step 16:__ Fit the identity control with `MLPClassifier(**{**fixed_settings, "activation": "identity"})` and inspect its collapsed affine prediction.

In [ ]:
identity_network = ....fit(
    moon_train_ready, moon_y_train)
collapsed_weights = identity_network.coefs_[0] @ identity_network.coefs_[1]
collapsed_bias = identity_network.intercepts_[0] @ identity_network.coefs_[1] + identity_network.intercepts_[1]
collapsed_probability = 1 / (1 + np.exp(-(moon_test_ready @ collapsed_weights + collapsed_bias)))
collapse_gap = np.abs(collapsed_probability[:, 0] - identity_network.predict_proba(moon_test_ready)[:, 1]).max()
print(f"largest identity-collapse gap: {collapse_gap:.2e}")
np.testing.assert_allclose(collapsed_probability[:, 0], identity_network.predict_proba(moon_test_ready)[:, 1])

**Collapsing the identity layers reproduces the fitted probabilities, with a largest gap of 2.22e-16.** The extra hidden layer therefore leaves an affine score before the logistic output.

__Step 17:__ Compare the fitted ReLU and identity boundaries on the moons.

In [ ]:
grid_x, grid_y = np.meshgrid(np.linspace(-2, 3, 180), np.linspace(-1.5, 2, 180))
grid = np.column_stack([grid_x.ravel(), grid_y.ravel()])
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
for ax, label, model in zip(axes, ["ReLU", "identity"], [toy_network, identity_network]):
    grid_probability = model.predict_proba(moon_scaler.transform(grid))[:, 1]
    ax.contourf(grid_x, grid_y, grid_probability.reshape(grid_x.shape), levels=12, cmap="coolwarm", alpha=0.35)
    ax.contour(grid_x, grid_y, grid_probability.reshape(grid_x.shape), levels=[0.5], colors="black")
    ax.scatter(moon_test[:, 0], moon_test[:, 1], c=moon_y_test, cmap="coolwarm", s=15, edgecolors="black", linewidths=0.3)
    score = accuracy_score(moon_y_test, model.predict(moon_test_ready))
    ax.set(title=f"{label}: test accuracy {score:.4f}", xlabel="First coordinate", ylabel="Second coordinate")
    print(f"{label} toy accuracy: {score:.4f}")
plt.tight_layout()
plt.show()

**ReLU reaches toy test accuracy 0.9667 and identity reaches 0.9167.** The ReLU boundary bends between the moons, while the identity boundary remains straight.

# <font color='#E8800A'>The MLP reference on the course data</font> <a class="anchor" id="reference"></a>
[Back to TOC](#toc)

The same default network now predicts the champions outcome. It is the reference: it is scored on the ten prepared folds, fitted on all the training athletes and logged before any setting changes.

<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html'>MLPClassifier(hidden_layer_sizes=(100,), activation='relu', solver='adam', alpha=0.0001, max_iter=200, random_state=33)</a>

**Definition:**
A multilayer perceptron classifier learns weights and biases connecting features, hidden units and class probabilities.

**Main Parameters:**
- `hidden_layer_sizes`: a tuple giving the units in each hidden layer.
- `activation`: the hidden transformation; identity, logistic, tanh or relu.
- `max_iter`: the iteration limit for the optimiser.
- `random_state`: fixes initial weights and shuffled training batches.

**Main Attributes (after fitting):**
- `coefs_`: one weight matrix per connection between layers.
- `intercepts_`: one bias vector per connection.
- `n_iter_`: iterations completed.
- `loss_`: the final training objective.
- `loss_curve_`: the objective recorded during fitting.

**Main Methods:**
- `fit(X, y)`: estimate weights and biases.
- `predict(X)`: return class labels.
- `predict_proba(X)`: return a probability for each class.
</div>

The feature scaling comparison also refits the preprocessing and feature selection.

__Step 18:__ Score the default network on the ten prepared folds.

In [ ]:
train_scores, validation_scores, fold_warnings = [], [], []
default_auc, default_balanced = [], []
for fold in tqdm(mlp_prepared, desc='mlp prepared', leave=True, position=0):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        fitted_fold = clone(MLPClassifier(**fixed_settings)).fit(fold["X_fit"], fold["y_fit"])
    # A fit that reaches the iteration limit raises a ConvergenceWarning; it is shown once here and counted below.
    for item in caught:
        warnings.warn(item.message, item.category)
    fold_warnings.append(len(caught) > 0)
    train_scores.append(f1_score(fold["y_fit"], fitted_fold.predict(fold["X_fit"])))
    validation_scores.append(f1_score(fold["y_score"], fitted_fold.predict(fold["X_score"])))
    default_auc.append(roc_auc_score(fold["y_score"], fitted_fold.predict_proba(fold["X_score"])[:, 1]))
    default_balanced.append(balanced_accuracy_score(fold["y_score"], fitted_fold.predict(fold["X_score"])))
print(f"{sum(fold_warnings)} of {len(fold_warnings)} folds raised a ConvergenceWarning")

__Step 19:__ Fit the same configuration on all training athletes.

In [ ]:
mlp_fitted = MLPClassifier(**fixed_settings).fit(Z_train, y_train)

__Step 20:__ Open the log with the default network's training, validation and held-out F1.

In [ ]:
mlp_log = []
record(mlp_log, {"frame": type(mlp_folds).__name__, "rows": len(y_train),
    "folds": mlp_folds.get_n_splits(), "name": "MLP (100,) relu, untuned defaults",
    "train": train_scores, "validation": validation_scores,
    "model": mlp_fitted}, Z_test, y_test, metric=f1_score, label="F1")
default_row = mlp_log[0].copy()

**The default network reaches cross-validated F1 0.8507 and held-out F1 0.8560, with training F1 0.9248.** All ten validation fits raise a ConvergenceWarning at the 200-iteration limit. The training score is higher than the score on unseen validation athletes.

# <font color='#E8800A'>MLP settings</font> <a class="anchor" id="mlp_settings"></a>
[Back to TOC](#toc)

The comparisons vary activation, layer sizes, iteration budget, batch size and feature scaling on the same ten training folds. The joint network combines the settings with the highest validation F1 in each comparison.

| Parameter | Meaning |
| --- | --- |
| feature scaling | Preparation with the feature scaler kept or removed. |
| `activation` | Function applied to each hidden weighted sum. |
| `hidden_layer_sizes` | Units in each hidden layer; its length is the number of hidden layers. |
| `max_iter` | Limit on the optimisation iterations. |
| `batch_size` | Rows in each minibatch update; the default `'auto'` is 200. |

[Parameter reference](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html)

### `MLP: activation`

The `activation` is the function applied to each **hidden** weighted sum $z$; it does not affect the output layer, which always applies the logistic function for binary classification. The default is `relu`.

| Activation | Formula | Range |
| --- | --- | --- |
| `identity` | $h(z) = z$ | $(-\infty,\,+\infty)$ |
| `logistic` | $h(z) = \dfrac{1}{1 + e^{-z}}$ | $(0,\,1)$ |
| `tanh` | $h(z) = \tanh(z)$ | $(-1,\,+1)$ |
| `relu` | $h(z) = \max(0,\,z)$ | $[0,\,+\infty)$ |

A nonlinear activation lets the hidden layer bend the decision boundary. With `identity` every hidden layer is an affine map, so no matter how many hidden layers the network has, the composition collapses to a single affine transformation. The sweep compares validation F1 across these activations while keeping the other network settings fixed. The candidate list is `["identity", "logistic", "tanh", "relu"]`.

__Step 21:__ Set the `activation` values.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
mlp_activation_values = ["identity", "logistic", "tanh", "relu"]
# One row per value will hold its scores, its seconds and its fitted model.
mlp_activation_rows = []


__Step 22:__ Score each candidate and keep its fitted model.

In [ ]:
for value in tqdm(mlp_activation_values, desc='mlp activation values', leave=True, position=0):
    started = time.perf_counter()
    settings = {**fixed_settings, "activation": value}
    train_scores, validation_scores, fold_warnings = [], [], []
    for fold in mlp_prepared:
        # A fit that stops at the iteration limit raises a ConvergenceWarning; it is counted here rather than shown.
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            fitted = MLPClassifier(**settings).fit(fold["X_fit"], fold["y_fit"])
        train_scores.append(f1_score(fold["y_fit"], fitted.predict(fold["X_fit"])))
        validation_scores.append(f1_score(fold["y_score"], fitted.predict(fold["X_score"])))
        fold_warnings.append(any(issubclass(w.category, ConvergenceWarning) for w in caught))
    # The fit on all the training rows is the model kept for the log and scored on the sealed test rows.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        deployed = MLPClassifier(**settings).fit(Z_train, y_train)
    # The label lists every setting that differs from the library default, and always the three that define the network.
    shown = {key: item for key, item in MLPClassifier(**settings).get_params().items() if item != MLPClassifier().get_params()[key] or key in ("activation", "hidden_layer_sizes", "max_iter")}
    mlp_activation_rows.append({"configuration": "MLPClassifier(" + ", ".join(f"{key}={item!r}" for key, item in sorted(shown.items())) + ")", "setting": str(value),
        "seconds": time.perf_counter() - started, "train_F1": np.mean(train_scores), "cv_F1": np.mean(validation_scores),
        "test_F1": f1_score(y_test, deployed.predict(Z_test)), "iterations": deployed.n_iter_, "loss": deployed.loss_,
        "fold warnings": sum(fold_warnings), "train": train_scores, "validation": validation_scores,
        "parameters": settings, "model": deployed})


__Step 23:__ Assemble and inspect the candidate scores.

In [ ]:
# The sweep table: one row per value, with the mean fold scores, the test score, the fit diagnostics and the seconds spent.
mlp_activation_sweep = pd.DataFrame(mlp_activation_rows)
mlp_activation_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "iterations", "loss", "fold warnings", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
mlp_activation_winner = mlp_activation_sweep.loc[mlp_activation_sweep["cv_F1"].idxmax()]
record(mlp_log, {"frame": type(mlp_folds).__name__, "rows": len(y_train), "folds": mlp_folds.get_n_splits(),
    "name": mlp_activation_winner["configuration"], "train": mlp_activation_winner["train"],
    "validation": mlp_activation_winner["validation"], "model": mlp_activation_winner["model"]},
    Z_test, y_test, "illustrative, untuned", metric=f1_score, label="F1")

__Step 24:__ Plot the training and validation F1 for `activation`.

In [ ]:
# Compare this parameter before moving to the next one.
parameter_positions = np.arange(len(mlp_activation_sweep))
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(parameter_positions, mlp_activation_sweep["train_F1"], marker="o", color=PLOT_ORANGE, label="training")
ax.plot(parameter_positions, mlp_activation_sweep["cv_F1"], marker="o", color=PLOT_BLUE, label="validation")
ax.set(title="activation", xticks=parameter_positions, xticklabels=mlp_activation_sweep["setting"], ylabel="F1")
ax.tick_params(axis="x", labelrotation=20)
ax.legend()
plt.tight_layout()
plt.show()

**Tanh has the highest validation F1, 0.8578, against logistic at 0.8511, ReLU at 0.8507 and identity at 0.8489.** ReLU fits the training folds most closely at F1 0.9248, while tanh reaches 0.8883. Identity raises no fold warnings, logistic warns on one fold, and tanh and ReLU warn on all ten.

### `MLP: hidden_layer_sizes`

`hidden_layer_sizes` lists the number of units in each hidden layer. For example, `(100,)` specifies one hidden layer of 100 units.

| `hidden_layer_sizes` | Units in each hidden layer |
| --- | --- |
| `(64, 64, 64)` | 64 units in the first, second and third hidden layers. |
| `(32, 64, 32)` | 32 units in the first hidden layer, 64 in the second and 32 in the third. |
| `(64, 32)` | 64 units in the first hidden layer and 32 in the second. |
| `(32, 64)` | 32 units in the first hidden layer and 64 in the second. |

Each hidden unit computes a weighted sum of the preceding layer's outputs, adds its bias and applies the activation function. Changing layer sizes changes the weight matrices and biases. A closer fit to training rows may not improve predictions on validation rows. The candidate list is `[(2,), (8,), (16,), (64,), (100,), (16, 16), (16, 16, 16), (64, 64, 64), (32, 64, 32), (64, 32), (32, 64)]`.

__Step 25:__ Set the `hidden_layer_sizes` values.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
mlp_hidden_layer_sizes_values = [(2,), (8,), (16,), (64,), (100,), (16, 16), (16, 16, 16), (64, 64, 64), (32, 64, 32), (64, 32), (32, 64)]
# One row per value will hold its scores, its seconds and its fitted model.
mlp_hidden_layer_sizes_rows = []


__Step 26:__ Score each candidate and keep its fitted model.

In [ ]:
for value in tqdm(mlp_hidden_layer_sizes_values, desc='mlp hidden layer sizes values', leave=True, position=0):
    started = time.perf_counter()
    settings = {**fixed_settings, "hidden_layer_sizes": value}
    train_scores, validation_scores, fold_warnings = [], [], []
    for fold in mlp_prepared:
        # A fit that stops at the iteration limit raises a ConvergenceWarning; it is counted here rather than shown.
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            fitted = MLPClassifier(**settings).fit(fold["X_fit"], fold["y_fit"])
        train_scores.append(f1_score(fold["y_fit"], fitted.predict(fold["X_fit"])))
        validation_scores.append(f1_score(fold["y_score"], fitted.predict(fold["X_score"])))
        fold_warnings.append(any(issubclass(w.category, ConvergenceWarning) for w in caught))
    # The fit on all the training rows is the model kept for the log and scored on the sealed test rows.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        deployed = MLPClassifier(**settings).fit(Z_train, y_train)
    # The label lists every setting that differs from the library default, and always the three that define the network.
    shown = {key: item for key, item in MLPClassifier(**settings).get_params().items() if item != MLPClassifier().get_params()[key] or key in ("activation", "hidden_layer_sizes", "max_iter")}
    mlp_hidden_layer_sizes_rows.append({"configuration": "MLPClassifier(" + ", ".join(f"{key}={item!r}" for key, item in sorted(shown.items())) + ")", "setting": str(value),
        "seconds": time.perf_counter() - started, "train_F1": np.mean(train_scores), "cv_F1": np.mean(validation_scores),
        "test_F1": f1_score(y_test, deployed.predict(Z_test)), "iterations": deployed.n_iter_, "loss": deployed.loss_,
        "fold warnings": sum(fold_warnings), "train": train_scores, "validation": validation_scores,
        "parameters": settings, "model": deployed})


__Step 27:__ Assemble and inspect the candidate scores.

In [ ]:
# The sweep table: one row per value, with the mean fold scores, the test score, the fit diagnostics and the seconds spent.
mlp_hidden_layer_sizes_sweep = pd.DataFrame(mlp_hidden_layer_sizes_rows)
mlp_hidden_layer_sizes_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "iterations", "loss", "fold warnings", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
mlp_hidden_layer_sizes_winner = mlp_hidden_layer_sizes_sweep.loc[mlp_hidden_layer_sizes_sweep["cv_F1"].idxmax()]
record(mlp_log, {"frame": type(mlp_folds).__name__, "rows": len(y_train), "folds": mlp_folds.get_n_splits(),
    "name": mlp_hidden_layer_sizes_winner["configuration"], "train": mlp_hidden_layer_sizes_winner["train"],
    "validation": mlp_hidden_layer_sizes_winner["validation"], "model": mlp_hidden_layer_sizes_winner["model"]},
    Z_test, y_test, "illustrative, untuned", metric=f1_score, label="F1")

__Step 28:__ Plot the training and validation F1 for `hidden_layer_sizes`.

In [ ]:
# Compare this parameter before moving to the next one.
parameter_positions = np.arange(len(mlp_hidden_layer_sizes_sweep))
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(parameter_positions, mlp_hidden_layer_sizes_sweep["train_F1"], marker="o", color=PLOT_ORANGE, label="training")
ax.plot(parameter_positions, mlp_hidden_layer_sizes_sweep["cv_F1"], marker="o", color=PLOT_BLUE, label="validation")
ax.set(title="hidden_layer_sizes", xticks=parameter_positions, xticklabels=mlp_hidden_layer_sizes_sweep["setting"], ylabel="F1")
ax.tick_params(axis="x", labelrotation=20)
ax.legend()
plt.tight_layout()
plt.show()

**Eight hidden units give the highest validation F1, 0.8541.** The `(64, 64, 64)` and `(32, 64, 32)` networks reach training F1 0.9957 and 0.9896, but validation F1 falls to 0.8152 and 0.8177. The narrowing `(64, 32)` and widening `(32, 64)` networks reach validation F1 0.8286 and 0.8367. These extra layers fit the training athletes more closely while predicting validation athletes less well at the same 200-iteration budget.

### `MLP: max_iter`

`max_iter` limits optimisation work; the reference allows 200 iterations. A `ConvergenceWarning` reports that the limit was reached before the stopping rule was satisfied. A fit without a warning may still predict poorly, so the table reports the warning count and training loss alongside training and validation F1. The candidate list is `[20, 200, 1000, 2000]`.

__Step 29:__ Set the `max_iter` values.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
mlp_max_iter_values = [20, 200, 1000, 2000]
# One row per value will hold its scores, its seconds and its fitted model.
mlp_max_iter_rows = []


__Step 30:__ Score each candidate and keep its fitted model.

In [ ]:
for value in tqdm(mlp_max_iter_values, desc='mlp max iter values', leave=True, position=0):
    started = time.perf_counter()
    settings = {**fixed_settings, "max_iter": value}
    train_scores, validation_scores, fold_warnings = [], [], []
    for fold in mlp_prepared:
        # A fit that stops at the iteration limit raises a ConvergenceWarning; it is counted here rather than shown.
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            fitted = MLPClassifier(**settings).fit(fold["X_fit"], fold["y_fit"])
        train_scores.append(f1_score(fold["y_fit"], fitted.predict(fold["X_fit"])))
        validation_scores.append(f1_score(fold["y_score"], fitted.predict(fold["X_score"])))
        fold_warnings.append(any(issubclass(w.category, ConvergenceWarning) for w in caught))
    # The fit on all the training rows is the model kept for the log and scored on the sealed test rows.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        deployed = MLPClassifier(**settings).fit(Z_train, y_train)
    # The label lists every setting that differs from the library default, and always the three that define the network.
    shown = {key: item for key, item in MLPClassifier(**settings).get_params().items() if item != MLPClassifier().get_params()[key] or key in ("activation", "hidden_layer_sizes", "max_iter")}
    mlp_max_iter_rows.append({"configuration": "MLPClassifier(" + ", ".join(f"{key}={item!r}" for key, item in sorted(shown.items())) + ")", "setting": str(value),
        "seconds": time.perf_counter() - started, "train_F1": np.mean(train_scores), "cv_F1": np.mean(validation_scores),
        "test_F1": f1_score(y_test, deployed.predict(Z_test)), "iterations": deployed.n_iter_, "loss": deployed.loss_,
        "fold warnings": sum(fold_warnings), "train": train_scores, "validation": validation_scores,
        "parameters": settings, "model": deployed})


__Step 31:__ Assemble and inspect the candidate scores.

In [ ]:
# The sweep table: one row per value, with the mean fold scores, the test score, the fit diagnostics and the seconds spent.
mlp_max_iter_sweep = pd.DataFrame(mlp_max_iter_rows)
mlp_max_iter_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "iterations", "loss", "fold warnings", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
mlp_max_iter_winner = mlp_max_iter_sweep.loc[mlp_max_iter_sweep["cv_F1"].idxmax()]
record(mlp_log, {"frame": type(mlp_folds).__name__, "rows": len(y_train), "folds": mlp_folds.get_n_splits(),
    "name": mlp_max_iter_winner["configuration"], "train": mlp_max_iter_winner["train"],
    "validation": mlp_max_iter_winner["validation"], "model": mlp_max_iter_winner["model"]},
    Z_test, y_test, "illustrative, untuned", metric=f1_score, label="F1")

__Step 32:__ Plot the training and validation F1 for `max_iter`.

In [ ]:
# Compare this parameter before moving to the next one.
parameter_positions = np.arange(len(mlp_max_iter_sweep))
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(parameter_positions, mlp_max_iter_sweep["train_F1"], marker="o", color=PLOT_ORANGE, label="training")
ax.plot(parameter_positions, mlp_max_iter_sweep["cv_F1"], marker="o", color=PLOT_BLUE, label="validation")
ax.set(title="max_iter", xticks=parameter_positions, xticklabels=mlp_max_iter_sweep["setting"], ylabel="F1")
ax.tick_params(axis="x", labelrotation=20)
ax.legend()
plt.tight_layout()
plt.show()

**The 200-iteration budget has the highest validation F1, 0.8507, against 0.8495 at 20 and 0.8230 at 2,000.** Training F1 rises from 0.8615 to 0.9900 across those budgets. All ten folds warn at 20 and 200 iterations, eight warn at 1,000, and none at 2,000; the all-training fit stops after 899 iterations at both larger budgets.

### `MLP: batch_size`

Adam updates the weights on one minibatch of rows at a time, and one pass through all the training rows is an epoch, so `batch_size` sets how many updates an epoch holds: 64 rows give many small updates and 512 rows give fewer, smoother ones. The default `'auto'` is a batch of 200 rows, so the reference is the 200 entry, and each `max_iter` iteration is one epoch. The candidate list is `[64, 200, 512]`.

__Step 33:__ Set the `batch_size` values.

In [ ]:
# The values of this one setting; every other setting stays at the reference.
mlp_batch_size_values = [64, 200, 512]
# One row per value will hold its scores, its seconds and its fitted model.
mlp_batch_size_rows = []


__Step 34:__ Score each candidate and keep its fitted model.

In [ ]:
for value in tqdm(mlp_batch_size_values, desc='mlp batch size values', leave=True, position=0):
    started = time.perf_counter()
    settings = {**fixed_settings, "batch_size": value}
    train_scores, validation_scores, fold_warnings = [], [], []
    for fold in mlp_prepared:
        # A fit that stops at the iteration limit raises a ConvergenceWarning; it is counted here rather than shown.
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            fitted = MLPClassifier(**settings).fit(fold["X_fit"], fold["y_fit"])
        train_scores.append(f1_score(fold["y_fit"], fitted.predict(fold["X_fit"])))
        validation_scores.append(f1_score(fold["y_score"], fitted.predict(fold["X_score"])))
        fold_warnings.append(any(issubclass(w.category, ConvergenceWarning) for w in caught))
    # The fit on all the training rows is the model kept for the log and scored on the sealed test rows.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        deployed = MLPClassifier(**settings).fit(Z_train, y_train)
    # The label lists every setting that differs from the library default, and always the three that define the network.
    shown = {key: item for key, item in MLPClassifier(**settings).get_params().items() if item != MLPClassifier().get_params()[key] or key in ("activation", "hidden_layer_sizes", "max_iter")}
    mlp_batch_size_rows.append({"configuration": "MLPClassifier(" + ", ".join(f"{key}={item!r}" for key, item in sorted(shown.items())) + ")", "setting": str(value),
        "seconds": time.perf_counter() - started, "train_F1": np.mean(train_scores), "cv_F1": np.mean(validation_scores),
        "test_F1": f1_score(y_test, deployed.predict(Z_test)), "iterations": deployed.n_iter_, "loss": deployed.loss_,
        "fold warnings": sum(fold_warnings), "train": train_scores, "validation": validation_scores,
        "parameters": settings, "model": deployed})


__Step 35:__ Assemble and inspect the candidate scores.

In [ ]:
# The sweep table: one row per value, with the mean fold scores, the test score, the fit diagnostics and the seconds spent.
mlp_batch_size_sweep = pd.DataFrame(mlp_batch_size_rows)
mlp_batch_size_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "iterations", "loss", "fold warnings", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
mlp_batch_size_winner = mlp_batch_size_sweep.loc[mlp_batch_size_sweep["cv_F1"].idxmax()]
record(mlp_log, {"frame": type(mlp_folds).__name__, "rows": len(y_train), "folds": mlp_folds.get_n_splits(),
    "name": mlp_batch_size_winner["configuration"], "train": mlp_batch_size_winner["train"],
    "validation": mlp_batch_size_winner["validation"], "model": mlp_batch_size_winner["model"]},
    Z_test, y_test, "illustrative, untuned", metric=f1_score, label="F1")

__Step 36:__ Plot the training and validation F1 for `batch_size`.

In [ ]:
# Compare this parameter before moving to the next one.
parameter_positions = np.arange(len(mlp_batch_size_sweep))
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(parameter_positions, mlp_batch_size_sweep["train_F1"], marker="o", color=PLOT_ORANGE, label="training")
ax.plot(parameter_positions, mlp_batch_size_sweep["cv_F1"], marker="o", color=PLOT_BLUE, label="validation")
ax.set(title="batch_size", xticks=parameter_positions, xticklabels=mlp_batch_size_sweep["setting"], ylabel="F1")
ax.tick_params(axis="x", labelrotation=20)
ax.legend()
plt.tight_layout()
plt.show()

**Batch size 512 gives the highest validation F1, 0.8568, against 0.8507 at 200 and 0.8469 at 64.** The smallest training fold has 2,560 athletes, so the default 'auto' uses 200 rows and reproduces the reference validation score exactly. All three settings warn on all ten folds, and the smaller batch reaches the highest training F1, 0.9448.

__Step 37:__ Confirm that the default `'auto'` batch is the 200 entry.

In [ ]:
# 'auto' means min(200, rows). Every fit here has more than 200 rows, so the batch_size=200 row repeats the reference fit.
smallest_fit = min(len(fold["y_fit"]) for fold in mlp_prepared)
print(f"rows in the smallest fit: {smallest_fit}, so 'auto' is a batch of {min(200, smallest_fit)}")
print("batch_size=200 scores the same validation F1 as the reference:", mlp_batch_size_sweep.set_index("setting").loc["200", "cv_F1"] == default_row["validation F1"])

__Step 38:__ Refit each batch size on the first prepared fold, stopped after k iterations, and read its training loss and its validation score.

In [ ]:
# Refit every batch size on the first prepared fold, stopped after k iterations; the validation rows of that fold score each refit and no test row is read.
curve_fold = mlp_prepared[0]
curve_budgets = [1, 3, 10, 30, 100, 300, 1000, 2000]
curve_points = {}
with warnings.catch_warnings():
    warnings.simplefilter("ignore", ConvergenceWarning)
    for curve_value in tqdm(mlp_batch_size_values, desc='mlp batch size values', leave=True, position=0):
        curve_rows = []
        for curve_k in curve_budgets:
            # A fit stopped after k iterations is the k-th iterate of the full fit, because the seed is fixed.
            curve_fit = MLPClassifier(**{**fixed_settings, "batch_size": curve_value, "max_iter": curve_k}).fit(curve_fold["X_fit"], curve_fold["y_fit"])
            curve_rows.append([curve_fit.n_iter_, curve_fit.loss_, f1_score(curve_fold["y_score"], curve_fit.predict(curve_fold["X_score"]))])
            if curve_fit.n_iter_ < curve_k:
                break
        curve_points[str(curve_value)] = np.array(curve_rows)
for curve_value, curve_rows in curve_points.items():
    print(f"batch size {curve_value}: loss {curve_rows[-1, 1]:.4f} and validation F1 {curve_rows[-1, 2]:.4f} after {int(curve_rows[-1, 0])} iterations")

__Step 39:__ Tabulate those readings.

In [ ]:
# One row per batch size and stopping point: the iterations reached, the training loss and the validation F1.
curve_table = pd.DataFrame([{"batch size": curve_value, "iterations": int(curve_row[0]), "training loss": curve_row[1], "validation F1": curve_row[2]}
                            for curve_value, curve_rows in curve_points.items() for curve_row in curve_rows])
display(curve_table.round(4))

__Step 40:__ Plot the training loss and the validation score of each batch size against the iteration.

In [ ]:
# One panel with two axes: the training loss is solid and read on the left, the validation F1 is dashed and read on the right.
curve_fig, curve_ax = plt.subplots(figsize=(9, 5.2))
curve_ax2 = curve_ax.twinx()
for (curve_value, curve_rows), (curve_colour, curve_marker) in zip(curve_points.items(), [("#33A367", "o"), ("#0D7A3E", "s"), ("#053D20", "^")]):
    curve_ax.plot(curve_rows[:, 0], curve_rows[:, 1], color=curve_colour, marker=curve_marker, ms=5, lw=2.2, label=f"batch size {curve_value}")
    curve_ax2.plot(curve_rows[:, 0], curve_rows[:, 2], color=curve_colour, marker=curve_marker, ms=5, lw=2.2, linestyle="--")
curve_ax.set_xscale("log")
curve_ax.set_ylim(bottom=0)
curve_ax.xaxis.set_major_formatter(plt.matplotlib.ticker.ScalarFormatter())
curve_ax.grid(axis="x", visible=False)
curve_ax2.grid(False)
curve_ax.legend(loc="upper right")
curve_ax.text(0.5, -0.2, "solid: training loss (left axis); dashed: validation F1 (right axis)", transform=curve_ax.transAxes, ha="center", va="top", fontsize=11)
curve_ax.set(xlabel="iteration (log scale)", ylabel="training loss\n(cross-entropy + L2 penalty)", title="Adam with three batch sizes")
curve_ax2.set_ylabel("validation F1, fold 1")
plt.tight_layout()
curve_fig.subplots_adjust(bottom=0.24)
plt.show()

**On fold 1, batch sizes 64, 200 and 512 end at validation F1 0.7810, 0.8052 and 0.7948 after 687, 1,328 and 1,791 iterations.** Their training losses are 0.0350, 0.0282 and 0.0402. Each curve refits that fold's training athletes and scores its validation athletes; the dashed scores rise early and then fall while training loss continues to decrease.

### `MLP: feature scaling`

The moons example uses standardised coordinates before fitting. This comparison measures the effect of feature scaling on the course data.

A copy of the recipe sets `scaler="none"`. `FoldPreprocessor` and `prepared_folds` reprocess the raw `X_train` with the same `mlp_folds` splitter; they do not inverse-transform the scaled folds. The remaining transformations and feature selection are fitted inside each training fold, so selected columns may differ. The preparation fitted on all training rows transforms `X_test` for assessment.

__Step 41:__ Prepare the training folds again without the scaler, keeping the rest of the recipe.

In [ ]:
with warnings.catch_warnings(record=True) as unscaled_preparation_warnings:
    warnings.simplefilter("always", ConvergenceWarning)
    with parallel_config(backend="loky", initializer=unscaled_preparation_warning_policy):
        unscaled_recipe = deepcopy(classification_recipe)
        unscaled_recipe["scaler"] = "none"
        unscaled_prep = FoldPreprocessor(unscaled_recipe).fit(X_train, y_train)
        unscaled_train = unscaled_prep.transform(X_train)
        unscaled_test = unscaled_prep.transform(X_test)
        with warnings.catch_warnings():
            unscaled_preparation_warning_policy()
            unscaled_prepared = prepared_folds(FoldPreprocessor(unscaled_recipe), mlp_folds,
                                               X_train, y_train, n_jobs=4)
unscaled_preparation_convergence = sum(isinstance(item.message, ConvergenceWarning) for item in unscaled_preparation_warnings)
print(f"Inherited unscaled preparation: {unscaled_preparation_convergence} convergence warnings from its LogisticRegression feature-selection fit; scientific settings are unchanged.")
print("The reported count is parent-process only; parallel preparation workers omit the same exact inherited 1000-iteration LogisticRegression notice. It is not a full-worker warning count.")


In [ ]:
# The values of this one setting; every other setting stays at the reference.
mlp_scale_values = ["scaler removed"]
# One row per value will hold its scores, its seconds and its fitted model.
mlp_scale_rows = []


__Step 42:__ Score each candidate and keep its fitted model.

In [ ]:
for value in tqdm(mlp_scale_values, desc='mlp scale values', leave=True, position=0):
    started = time.perf_counter()
    settings = fixed_settings
    train_scores, validation_scores, fold_warnings = [], [], []
    for fold in unscaled_prepared:
        # A fit that stops at the iteration limit raises a ConvergenceWarning; it is counted here rather than shown.
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            fitted = MLPClassifier(**settings).fit(fold["X_fit"], fold["y_fit"])
        train_scores.append(f1_score(fold["y_fit"], fitted.predict(fold["X_fit"])))
        validation_scores.append(f1_score(fold["y_score"], fitted.predict(fold["X_score"])))
        fold_warnings.append(any(issubclass(w.category, ConvergenceWarning) for w in caught))
    # The fit on all the training rows is the model kept for the log and scored on the sealed test rows.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        deployed = MLPClassifier(**settings).fit(unscaled_train, y_train)
    # The label lists every setting that differs from the library default, and always the three that define the network.
    shown = {key: item for key, item in MLPClassifier(**settings).get_params().items() if item != MLPClassifier().get_params()[key] or key in ("activation", "hidden_layer_sizes", "max_iter")}
    mlp_scale_rows.append({"configuration": "MLPClassifier(" + ", ".join(f"{key}={item!r}" for key, item in sorted(shown.items())) + ") without the scaler", "setting": str(value),
        "seconds": time.perf_counter() - started, "train_F1": np.mean(train_scores), "cv_F1": np.mean(validation_scores),
        "test_F1": f1_score(y_test, deployed.predict(unscaled_test)), "iterations": deployed.n_iter_, "loss": deployed.loss_,
        "fold warnings": sum(fold_warnings), "train": train_scores, "validation": validation_scores,
        "parameters": settings, "model": deployed})


__Step 43:__ Assemble and inspect the candidate scores.

In [ ]:
# The sweep table: one row per value, with the mean fold scores, the test score, the fit diagnostics and the seconds spent.
mlp_scale_sweep = pd.DataFrame(mlp_scale_rows)
mlp_scale_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "iterations", "loss", "fold warnings", "seconds"]].round(4)

In [ ]:
# The sweep's winner joins the log.
mlp_scale_winner = mlp_scale_sweep.loc[mlp_scale_sweep["cv_F1"].idxmax()]
record(mlp_log, {"frame": type(mlp_folds).__name__, "rows": len(y_train), "folds": mlp_folds.get_n_splits(),
    "name": mlp_scale_winner["configuration"], "train": mlp_scale_winner["train"],
    "validation": mlp_scale_winner["validation"], "model": mlp_scale_winner["model"]},
    unscaled_test, y_test, "illustrative, untuned", metric=f1_score, label="F1")

__Step 44:__ Plot the training and validation F1 for `scale`.

In [ ]:
parameter_table = pd.concat([pd.DataFrame([{"setting": "scaler kept", "train_F1": default_row["train F1"], "cv_F1": default_row["validation F1"]}]), mlp_scale_sweep], ignore_index=True)
# Compare this parameter before moving to the next one.
parameter_positions = np.arange(len(parameter_table))
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(parameter_positions, parameter_table["train_F1"], marker="o", color=PLOT_ORANGE, label="training")
ax.plot(parameter_positions, parameter_table["cv_F1"], marker="o", color=PLOT_BLUE, label="validation")
ax.set(title="scale", xticks=parameter_positions, xticklabels=parameter_table["setting"], ylabel="F1")
ax.tick_params(axis="x", labelrotation=20)
ax.legend()
plt.tight_layout()
plt.show()

**Removing the scaler lowers validation F1 from 0.8507 to 0.8331 and test F1 from 0.8560 to 0.8382.** The unscaled preparation records five parent-process convergence warnings from its LogisticRegression feature-selection fit; that count excludes the workers' notices. The unscaled network itself warns on four validation folds. The joint network therefore keeps the inherited scaler.

__Step 45:__ Combine the best value of every sweep, read from the sweep tables, into the joint network.

### The joint network

The joint network uses the activation, hidden layer sizes, iteration budget and batch size with the highest validation F1 in their respective tables. It removes the feature scaler only if the unscaled preparation scores higher than the reference. The reference value is available in every sweep.

In [ ]:
# The best validation value of each sweep, read from its table.
joint_sweeps = {"activation": mlp_activation_sweep, "hidden_layer_sizes": mlp_hidden_layer_sizes_sweep,
                "max_iter": mlp_max_iter_sweep, "batch_size": mlp_batch_size_sweep}
joint_best = {setting: table.loc[table["cv_F1"].idxmax()] for setting, table in joint_sweeps.items()}
joint_settings = {**fixed_settings, **{setting: row["parameters"][setting] for setting, row in joint_best.items()}}
# The scale control has one row, the preparation without the scaler; it joins only if it scored higher than the reference.
joint_unscaled = mlp_scale_sweep["cv_F1"].iat[0] > default_row["validation F1"]
joint_folds, joint_train, joint_test, joint_prep = ((unscaled_prepared, unscaled_train, unscaled_test, unscaled_prep) if joint_unscaled
                                                    else (mlp_prepared, Z_train, Z_test, mlp_prep))
reference_settings = MLPClassifier(**fixed_settings).get_params()
joint_choices = pd.DataFrame({
    "reference": [reference_settings[setting] for setting in joint_sweeps] + ["scaler kept"],
    "joint value": [joint_settings[setting] for setting in joint_sweeps] + ["scaler removed" if joint_unscaled else "scaler kept"],
    "validation F1": [row["cv_F1"] for row in joint_best.values()] + [max(mlp_scale_sweep["cv_F1"].iat[0], default_row["validation F1"])]},
    index=[*joint_sweeps, "scale"])
joint_choices.round(4)

__Step 46:__ Score the joint network on the ten prepared folds, as every sweep row was scored.

In [ ]:
joint_train_scores, joint_validation_scores, joint_fold_warnings = [], [], []
joint_auc, joint_balanced = [], []
for fold in tqdm(joint_folds, desc='joint folds', leave=True, position=0):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        fitted_fold = MLPClassifier(**joint_settings).fit(fold["X_fit"], fold["y_fit"])
    joint_train_scores.append(f1_score(fold["y_fit"], fitted_fold.predict(fold["X_fit"])))
    joint_validation_scores.append(f1_score(fold["y_score"], fitted_fold.predict(fold["X_score"])))
    joint_auc.append(roc_auc_score(fold["y_score"], fitted_fold.predict_proba(fold["X_score"])[:, 1]))
    joint_balanced.append(balanced_accuracy_score(fold["y_score"], fitted_fold.predict(fold["X_score"])))
    joint_fold_warnings.append(any(issubclass(w.category, ConvergenceWarning) for w in caught))
print(f"{sum(joint_fold_warnings)} of {len(joint_fold_warnings)} folds raised a ConvergenceWarning")

__Step 47:__ Fit the joint network on all training athletes and log it.

In [ ]:
joint_fitted = MLPClassifier(**joint_settings).fit(joint_train, y_train)
joint_name = (f"MLP joint of the sweeps: {joint_settings['hidden_layer_sizes']} {joint_settings['activation']}, "
              f"max_iter={joint_settings['max_iter']}, batch_size={joint_settings['batch_size']}, scaler {'removed' if joint_unscaled else 'kept'}")
record(mlp_log, {"frame": type(mlp_folds).__name__, "rows": len(y_train), "folds": mlp_folds.get_n_splits(),
    "name": joint_name, "train": joint_train_scores, "validation": joint_validation_scores, "model": joint_fitted},
    joint_test, y_test, "chosen on these folds", metric=f1_score, label="F1")
joint_row = mlp_log[-1].copy()

**The joint network combines tanh, eight hidden units, 200 iterations, batch size 512 and the inherited scaler, reaching validation F1 0.8523 and test F1 0.8509.** The combination scores below tanh's one-setting validation F1 of 0.8578, so individually better settings need not work better together. Its cross-validated score is optimistic because the folds that chose each setting also score the combination. Week 13's nested cross-validation measures a search on outer validation rows that did not choose its settings.

__Step 48:__ Read the fitted arrays and the activation function of each layer, and choose the units to draw.

In [ ]:
# The fitted network's own arrays. A layer wider than six units shows its first five units, an ellipsis and its last unit.
net_model = joint_fitted
net_weights, net_biases = net_model.coefs_, net_model.intercepts_
net_sizes = [net_weights[0].shape[0]] + [weight.shape[1] for weight in net_weights]
net_shown = [list(range(size)) if size <= 6 else [*range(5), None, size - 1] for size in net_sizes]
# The activation of each layer is read from the fitted model: `activation` for the hidden layers and `out_activation_` for the output.
net_activations = ["none"] + [net_model.activation] * (len(net_sizes) - 2) + [net_model.out_activation_]
net_symbol = "p"
net_layers = pd.DataFrame({"units": net_sizes, "units drawn": [len([unit for unit in shown if unit is not None]) for shown in net_shown],
                           "activation": net_activations},
                          index=["input"] + [f"hidden {layer}" for layer in range(1, len(net_sizes) - 1)] + ["output"])
display(net_layers)

__Step 49:__ Place the drawn units and name their nodes.

In [ ]:
# One row per unit drawn, and one row per connection between units drawn in neighbouring layers.
net_gap = 4
net_prefix = ["x"] + (["h"] if len(net_sizes) == 3 else [f"h{layer}_" for layer in range(1, len(net_sizes) - 1)])
net_math = ["x"] + (["h"] if len(net_sizes) == 3 else [f"h^{{({layer})}}" for layer in range(1, len(net_sizes) - 1)])
net_node_rows = []
for layer, slots in enumerate(net_shown):
    last = layer == len(net_sizes) - 1
    for slot, unit in enumerate(slots):
        net_node_rows.append({"layer": layer, "unit": -1 if unit is None else unit, "x": layer * net_gap, "y": 1.2 * ((len(slots) - 1) / 2 - slot),
                              "name": "output" if last else f"{net_prefix[layer]}{unit + 1}" if unit is not None else "",
                              "tex": f"${net_symbol}$" if last else f"${net_math[layer]}_{{{unit + 1}}}$" if unit is not None else ""})
net_nodes = pd.DataFrame(net_node_rows)
net_circles = net_nodes[net_nodes["unit"] >= 0]


__Step 50:__ Extract the fitted weights connecting the drawn units.

In [ ]:
net_link_rows = []
for layer, weights in enumerate(net_weights):
    for source in net_circles[net_circles["layer"] == layer].itertuples():
        for target in net_circles[net_circles["layer"] == layer + 1].itertuples():
            net_link_rows.append({"from": source.name, "to": target.name, "weight": weights[source.unit, target.unit],
                                  "x0": source.x, "y0": source.y, "x1": target.x, "y1": target.y})
net_links = pd.DataFrame(net_link_rows)
net_largest = max(np.abs(weight).max() for weight in net_weights)


__Step 51:__ Label each layer with its fitted activation and array shapes.

In [ ]:
# The text of the figure: a title for each layer with its activation function, and the shapes of each pair of arrays.
net_hidden_formula = {"relu": r"\max(0,z)", "tanh": r"\tanh(z)", "logistic": r"\sigma(z)", "identity": "z"}[net_model.activation]
net_output_formula = {"logistic": r"\sigma(a)", "identity": "a"}[net_model.out_activation_]
net_hidden_names = ["hidden layer" if len(net_sizes) == 3 else f"hidden layer {layer}" for layer in range(1, len(net_sizes) - 1)]
net_titles = ([f"inputs $x$\n{net_sizes[0]} units"]
              + [f"{name}, {size} units\n{net_model.activation}: $h={net_hidden_formula}$" for name, size in zip(net_hidden_names, net_sizes[1:-1])]
              + [f"output\n{net_model.out_activation_}: ${net_symbol}={net_output_formula}$" + ""])
net_captions = [f"$W_{{{k}}}$ ({w.shape[0]}×{w.shape[1]}), $b_{{{k}}}$ ({len(b)})" for k, (w, b) in enumerate(zip(net_weights, net_biases), 1)]
net_face = ["#EDEDED"] + ["#FFF3C4"] * (len(net_sizes) - 2) + ["#DFF5E8"]
print(f"{len(net_links)} connections drawn; largest absolute weight of the network {net_largest:.3f}")
# The ten largest of the connections drawn.
display(net_links.assign(size=net_links["weight"].abs()).nlargest(10, "size")[["from", "to", "weight"]].round(3).reset_index(drop=True))

__Step 52:__ Draw the network from that table.

In [ ]:
net_blue, net_orange = "#1779A8", "#C96A00"
net_fig, net_ax = plt.subplots(figsize=(min(14, 2.6 * len(net_sizes) + 2), 7.4))
for link in net_links.itertuples():
    net_size = np.abs(link.weight) / net_largest
    net_ax.plot([link.x0, link.x1], [link.y0, link.y1], color=net_blue if link.weight > 0 else net_orange,
                lw=0.4 + 4.6 * net_size, alpha=0.12 + 0.8 * net_size, solid_capstyle="round", zorder=1)
net_ax.scatter(net_circles["x"], net_circles["y"], s=560, facecolor=[net_face[layer] for layer in net_circles["layer"]],
               edgecolor="#222222", linewidth=1.2, zorder=3)
for node in net_nodes.itertuples():
    net_ax.text(node.x, node.y, node.tex if node.unit >= 0 else r"$\vdots$", ha="center", va="center",
                fontsize=11 if node.unit >= 0 else 18, zorder=4)
net_top, net_bottom = net_nodes["y"].max(), net_nodes["y"].min()
for layer, title in enumerate(net_titles):
    net_ax.text(layer * net_gap, net_top + 0.9, title, ha="center", va="bottom", fontweight="bold", fontsize=12)
for gap, caption in enumerate(net_captions):
    net_ax.text((gap + 0.5) * net_gap, net_bottom - 1.0, caption, ha="center", va="top", fontsize=12)
# Two keys: the sign, and three sample widths.
net_sign_key = net_ax.legend(handles=[net_ax.plot([], [], color=net_blue, lw=3, label="positive")[0], net_ax.plot([], [], color=net_orange, lw=3, label="negative")[0]],
                             title="weight sign", loc="lower left", bbox_to_anchor=(0.0, -0.06), ncol=2, frameon=False, fontsize=11, title_fontsize=11)
net_ax.add_artist(net_sign_key)
net_ax.legend(handles=[net_ax.plot([], [], color="#555555", lw=0.4 + 4.6 * size, label=f"{size * net_largest:.2f}")[0] for size in (0.25, 0.5, 1.0)],
              title="weight size", loc="lower right", bbox_to_anchor=(1.0, -0.06), ncol=3, frameon=False, fontsize=11, title_fontsize=11)
net_ax.set(xlim=(-1, net_gap * (len(net_sizes) - 1) + 1.4), ylim=(net_bottom - 2.8, net_top + 3.4), title="The joint network")
net_ax.axis("off")
plt.tight_layout()
plt.show()

**The joint network connects 28 inputs to eight tanh units and one logistic output; its largest absolute weight is 1.244.** The diagram reads the activation names from the fitted model. Its ellipses mark inputs and hidden units that are present in the fitted arrays but omitted from the drawing.

# <font color='#E8800A'>The course board</font> <a class="anchor" id="board"></a>
[Back to TOC](#toc)

The board reports the joint network with scores from previous course weeks. The baseline predicts the training majority class for every row.

__Step 53:__ Read the recorded training, cross-validation and test scores together.

In [ ]:
comparison = pd.DataFrame(mlp_log).rename(columns={"validation F1": "cv F1"})
display(comparison[["model", "train F1", "cv F1", "test F1"]].round(4))

__Step 54:__ Build the course board with the joint network and the saved earlier results.

In [ ]:
results_dir = "../../results"
floor = f1_score(y_test, np.full(len(y_test), y_train.mode().iat[0]))
board = Scoreboard("champions outcome", "F1", floor=floor, context=result_context(
    "../../data/interim/champions.csv", classification_recipe_log,
    y_train.index.tolist(), y_test.index.tolist(),
    outer_folds=outer_folds(mlp_folds, X_train, y_train)))
for week, model, cv, test in results_before(12, "classification", results_dir,
                                          metric=board.metric, context=board.context):
    board.record(week, model, cv, test)
board.record(12, joint_row["model"], joint_row["validation F1"], joint_row["test F1"])
board.save(results_dir, 12, "classification",
           baseline_cv=default_row["validation F1"], baseline_test=default_row["test F1"])
print(board.table())
# Package the already fitted preparation and joint network without fitting them again.
chosen_model = Pipeline([("preparation", joint_prep), ("model", joint_fitted)])
joblib.dump(chosen_model, f"{results_dir}/week_12_classification_model.joblib")
# The joint network's settings, for the week that starts from it.
joint_file = Path(results_dir) / "mlp" / "joint_classification.json"
joint_file.parent.mkdir(parents=True, exist_ok=True)
joint_file.write_text(json.dumps({"model": joint_row["model"], "metric": "F1", "cv": joint_row["validation F1"], "test": joint_row["test F1"],
                                  "settings": {**joint_settings, "hidden_layer_sizes": list(joint_settings["hidden_layer_sizes"])},
                                  "scaler": "removed" if joint_unscaled else "kept",
                                  "fitted_model": "../week_12_classification_model.joblib"}, indent=1) + "\n", encoding="utf-8")

# Companion metrics describe the same fitted models; F1 alone ranks candidates.
companion_metrics = pd.DataFrame([
    {"model": "scikit-learn defaults", "CV F1": default_row["validation F1"], "test F1": default_row["test F1"], "CV ROC-AUC": np.mean(default_auc), "test ROC-AUC": roc_auc_score(y_test, mlp_fitted.predict_proba(Z_test)[:, 1]), "CV balanced accuracy": np.mean(default_balanced), "test balanced accuracy": balanced_accuracy_score(y_test, mlp_fitted.predict(Z_test))},
    {"model": "joint network", "CV F1": joint_row["validation F1"], "test F1": joint_row["test F1"], "CV ROC-AUC": np.mean(joint_auc), "test ROC-AUC": roc_auc_score(y_test, joint_fitted.predict_proba(joint_test)[:, 1]), "CV balanced accuracy": np.mean(joint_balanced), "test balanced accuracy": balanced_accuracy_score(y_test, joint_fitted.predict(joint_test))},
])
display(companion_metrics)

**The joint network enters at cross-validated F1 0.8523 and held-out F1 0.8509, above the majority-class floor of 0.7529.** Its validation ROC-AUC is 0.8741 and balanced accuracy 0.7909. The default network scores 0.8507 and 0.8560 in F1, so the combination improves validation F1 but lowers test F1. The saved forest and stacking ensemble remain higher on both estimates.

# <font color='#E8800A'>Key takeaways</font> <a class="anchor" id="takeaways"></a>
[Back to TOC](#toc)

1. Weight matrices and bias vectors determine the forward pass; the manual calculation checks the library probability.
2. Identity hidden layers collapse into one affine score, a weighted sum plus bias. Nonlinear activations give the network a different response in different input regions.
3. Activation, depth and width, scale, iteration budget and batch size change different parts of the calculation, and each sweep shows what one of them does.
4. The joint network combines the best value of every sweep and is the board row. Its cross-validated score is optimistic, because the folds that chose each value also score the combination; Week 13's nested cross-validation measures a search on outer validation rows.

# <font color='#E8800A'>Exit ticket</font> <a class="anchor" id="exit"></a>
[Back to TOC](#toc)

Explain why replacing ReLU with identity removes the curved boundary, even when the hidden layer has many units.

In [ ]:
# Write your answer below. No Python is required.
#


# <font color='#E8800A'>References</font> <a class="anchor" id="references"></a>
[Back to TOC](#toc)

- Goodfellow, I., Bengio, Y. and Courville, A. (2016). *Deep Learning*, [Chapter 6](https://www.deeplearningbook.org/contents/mlp.html).
- scikit-learn, [Neural network models](https://scikit-learn.org/stable/modules/neural_networks_supervised.html).